In [15]:
import numpy as np
import json
import re
from sentence_transformers import SentenceTransformer
from sklearn.metrics.pairwise import cosine_similarity

# -----------------------------
# Load BERT Model
# -----------------------------
model = SentenceTransformer('all-MiniLM-L6-v2')

# **FAQ Data**

In [18]:
with open("semantic_faq.json", "r", encoding="utf-8") as file:
    faq_data = json.load(file)

# **Preparing Training Data**

In [21]:
questions = []
answers = []

for item in faq_data:
    for q in item["questions"]:
        questions.append(q)
        answers.append(item["answer"])

# -----------------------------
# Create Embeddings
# -----------------------------

question_embeddings = model.encode(questions)

# -----------------------------
# Get Best Answer
# -----------------------------

def get_answer(user_input, threshold=0.5):

    # Preprocessing
    user_input = user_input.lower().strip()
    user_input = re.sub(r"[^\w\s]", "", user_input)

    # Convert user input to embedding
    user_embedding = model.encode([user_input])

    # Similarity check
    similarities = cosine_similarity(
        user_embedding,
        question_embeddings
    )

    # Best match
    best_match_idx = np.argmax(similarities)

    best_score = similarities[0][best_match_idx]

    # Confidence threshold
    if best_score < threshold:
        return "Sorry, I don't have information about that."

    return answers[best_match_idx]

# -----------------------------
# Chat Loop
# -----------------------------

print("🤖 Semantic FAQ Chatbot Started! Type 'exit' to stop.\n")

while True:

    user_input = input("You: ")

    if user_input.lower() == "exit":
        print("Bot: Goodbye!")
        break

    response = get_answer(user_input)

    print("Bot:", response)

🤖 Semantic FAQ Chatbot Started! Type 'exit' to stop.

Bot: Hello! How can I help you?
Bot: Sorry, I don't have information about that.
Bot: Hello! How can I help you?
Bot: The college provides well-equipped labs, a central library, indoor and outdoor sports grounds, a gym, and a multi-cuisine cafeteria.
Bot: The college offers B.Tech in Computer Science, Mechanical, Civil, Electrical, Electronics & Communication; MBA in Finance, Marketing, HR; BBA; BCA; M.Tech in AI & Data Science, Structural Engineering.
Bot: Fee structure: B.Tech $10k/yr, M.Tech $12k/yr, MBA $15k/yr, BBA/BCA $8k/yr, Hostel $2.5k/semester.
Bot: Yes, the college provides separate hostel facilities for boys and girls with mess, Wi-Fi, and 24/7 security.
Bot: Sorry, I don't have information about that.
Bot: Yes, the college has a central library with books, journals, and digital resources.
Bot: Goodbye!


In [22]:
import pickle

with open("chatbot_embeddings.pkl", "wb") as f:
    pickle.dump(question_embeddings, f)

with open("chatbot_data.pkl", "wb") as f:
    pickle.dump((questions, answers), f)